# Moving-Average Trend-Following Backtest
## Gold (GC=F) & Crude Oil (CL=F)

This notebook backtests a simple SMA crossover strategy on commodity futures:
- **Signal**: Long when short SMA > long SMA, else flat
- **Train/Test split**: First 7 years / last 3 years
- **Parameter tuning**: Grid search on train set only
- **Transaction cost**: 0.05% per trade


In [ ]:
# Install yfinance if running on Colab
!pip install yfinance -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

# Reproducibility
np.random.seed(42)


## 1. Load Data
Download ~10 years of daily adjusted close prices from Yahoo Finance.


In [ ]:
def load_data(ticker, period="10y"):
    """Download daily adjusted close prices from Yahoo Finance."""
    df = yf.download(ticker, period=period, auto_adjust=True)
    close = df["Close"].dropna().squeeze()
    close.name = ticker
    print(f"{ticker}: {close.index[0].date()} to {close.index[-1].date()}, "
          f"{len(close)} rows")
    return close

# Download gold and crude oil
gold = load_data("GC=F")
oil = load_data("CL=F")


## 2. Generate Signals
Long (1) when the short SMA is above the long SMA, otherwise flat (0).  
The signal is shifted by one day to prevent look-ahead bias.


In [ ]:
def generate_signals(prices, short_window, long_window):
    """
    Signal = 1 (long) when short SMA > long SMA, else 0 (flat).
    Shift by 1 day to avoid look-ahead bias: today's signal
    is based on yesterday's closing SMAs, so we only trade on
    information we actually had at market open.
    """
    sma_short = prices.rolling(window=short_window).mean()
    sma_long = prices.rolling(window=long_window).mean()
    raw_signal = (sma_short > sma_long).astype(int)
    signal = raw_signal.shift(1)  # No look-ahead bias
    return signal


## 3. Backtest Engine
Compute daily strategy returns with 0.05% transaction cost on each position change.


In [ ]:
def backtest(prices, signal, cost_bps=5):
    """
    Compute strategy returns with transaction costs.
    cost_bps: one-way cost in basis points (5 = 0.05%).
    """
    daily_returns = prices.pct_change()

    # Strategy return = market return * position (1 or 0)
    strategy_returns = daily_returns * signal

    # Deduct cost each time position changes
    trades = signal.diff().abs()
    cost = trades * (cost_bps / 10_000)
    strategy_returns = strategy_returns - cost

    # Cumulative equity curves (start at 1)
    equity_strategy = (1 + strategy_returns).cumprod()
    equity_benchmark = (1 + daily_returns).cumprod()

    result = pd.DataFrame({
        "price": prices,
        "signal": signal,
        "daily_ret": daily_returns,
        "strategy_ret": strategy_returns,
        "equity_strategy": equity_strategy,
        "equity_benchmark": equity_benchmark,
        "trades": trades,
    })
    return result


## 4. Performance Metrics
Annualized return, volatility, Sharpe ratio (rf=0, 252 days), max drawdown, win rate, trades.


In [ ]:
def compute_metrics(returns, label=""):
    """Compute annualized performance metrics. 252 trading days, rf=0."""
    returns = returns.dropna()
    n_days = len(returns)
    if n_days == 0:
        return {}

    total_return = (1 + returns).prod() - 1
    ann_return = (1 + total_return) ** (252 / n_days) - 1
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol != 0 else 0.0

    cum = (1 + returns).cumprod()
    running_max = cum.cummax()
    drawdown = (cum - running_max) / running_max
    max_dd = drawdown.min()

    win_rate = (returns[returns != 0] > 0).mean() if (returns != 0).any() else 0.0

    return {
        "Ann. Return": f"{ann_return:.2%}",
        "Ann. Volatility": f"{ann_vol:.2%}",
        "Sharpe Ratio": f"{sharpe:.2f}",
        "Max Drawdown": f"{max_dd:.2%}",
        "Win Rate": f"{win_rate:.2%}",
    }


def count_trades(signal):
    """Count number of position changes."""
    return int(signal.diff().abs().sum())


## 5. Train/Test Split & Grid Search
- **Train**: first 7 years
- **Test**: last 3 years
- Grid search short window in [20, 50, 100] and long window in [100, 150, 200] (short < long)
- Pick the best pair by Sharpe ratio on training data only


In [ ]:
def split_train_test(prices, train_years=7):
    """Split into first train_years (train) and the rest (test)."""
    start = prices.index[0]
    cutoff = start + pd.DateOffset(years=train_years)
    train = prices[prices.index < cutoff]
    test = prices[prices.index >= cutoff]
    return train, test, cutoff


def grid_search(prices_train, short_windows, long_windows):
    """
    Search over (short, long) SMA pairs on training data.
    Returns the pair with the highest Sharpe ratio.
    """
    best_sharpe = -np.inf
    best_params = (50, 200)
    results_log = []

    for sw in short_windows:
        for lw in long_windows:
            if sw >= lw:
                continue  # short must be < long
            sig = generate_signals(prices_train, sw, lw)
            bt = backtest(prices_train, sig)
            ret = bt["strategy_ret"].dropna()
            n = len(ret)
            if n == 0:
                continue
            total = (1 + ret).prod() - 1
            ann_ret = (1 + total) ** (252 / n) - 1
            ann_vol = ret.std() * np.sqrt(252)
            sharpe = ann_ret / ann_vol if ann_vol != 0 else 0
            results_log.append({"short": sw, "long": lw,
                                "sharpe": round(sharpe, 4),
                                "ann_ret": f"{ann_ret:.2%}"})
            if sharpe > best_sharpe:
                best_sharpe = sharpe
                best_params = (sw, lw)

    print("Grid search results (train set):")
    for r in results_log:
        marker = " <-- best" if (r["short"] == best_params[0] and
                                 r["long"] == best_params[1]) else ""
        print(f"  SMA({r['short']:>3}/{r['long']:>3})  "
              f"Sharpe={r['sharpe']:>7.4f}  Ann.Ret={r['ann_ret']}{marker}")
    print(f"\nBest: SMA({best_params[0]}/{best_params[1]}) "
          f"Sharpe={best_sharpe:.4f}")
    return best_params


## 6. Plotting
Equity curves (strategy vs buy-and-hold) with train/test boundary, plus drawdown chart.


In [ ]:
def plot_results(bt_full, cutoff, ticker, short_w, long_w):
    """Plot equity curves and drawdown chart."""
    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]})

    # Equity curves
    ax = axes[0]
    ax.plot(bt_full.index, bt_full["equity_strategy"],
            label=f"SMA({short_w}/{long_w}) Strategy", linewidth=1.3)
    ax.plot(bt_full.index, bt_full["equity_benchmark"],
            label="Buy & Hold", linewidth=1.3, alpha=0.8)
    ax.axvline(cutoff, color="grey", linestyle="--", alpha=0.7,
               label="Train / Test boundary")
    ax.set_ylabel("Cumulative Return (starting at 1)")
    ax.set_title(f"{ticker} - SMA({short_w}/{long_w}) Strategy vs Buy & Hold")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)

    # Drawdown
    ax2 = axes[1]
    cum = bt_full["equity_strategy"]
    dd = (cum - cum.cummax()) / cum.cummax()
    ax2.fill_between(dd.index, dd, 0, color="red", alpha=0.35,
                     label="Strategy Drawdown")
    cum_bh = bt_full["equity_benchmark"]
    dd_bh = (cum_bh - cum_bh.cummax()) / cum_bh.cummax()
    ax2.fill_between(dd_bh.index, dd_bh, 0, color="blue", alpha=0.15,
                     label="Buy & Hold Drawdown")
    ax2.axvline(cutoff, color="grey", linestyle="--", alpha=0.7)
    ax2.set_ylabel("Drawdown")
    ax2.set_xlabel("Date")
    ax2.legend(loc="lower left")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


## 7. Run the Backtest
Run the full pipeline for each asset: load data, split, grid search, backtest, compute metrics, and plot.


In [ ]:
def run_asset(ticker):
    """Full pipeline for one asset."""
    print(f"\n{'='*60}")
    print(f"  {ticker}")
    print(f"{'='*60}")

    # Load data
    prices = load_data(ticker)

    # Train / test split
    train, test, cutoff = split_train_test(prices, train_years=7)
    print(f"Train: {train.index[0].date()} to {train.index[-1].date()} ({len(train)} rows)")
    print(f"Test:  {test.index[0].date()} to {test.index[-1].date()} ({len(test)} rows)")

    # Grid search on train data only
    short_windows = [20, 50, 100]
    long_windows = [100, 150, 200]
    best_short, best_long = grid_search(train, short_windows, long_windows)

    # Backtest full series with best params
    signal = generate_signals(prices, best_short, best_long)
    bt_full = backtest(prices, signal)

    # Split backtest results for metrics
    bt_train = bt_full[bt_full.index < cutoff]
    bt_test = bt_full[bt_full.index >= cutoff]

    # Compute metrics for train and test
    metrics = {}
    for period_name, bt_slice in [("Train", bt_train), ("Test", bt_test)]:
        for src, label in [("strategy_ret", "Strategy"), ("daily_ret", "Buy & Hold")]:
            key = f"{ticker} | {period_name} | {label}"
            m = compute_metrics(bt_slice[src], label)
            m["Trades"] = count_trades(bt_slice["signal"]) if label == "Strategy" else "-"
            metrics[key] = m

    # Plot
    plot_results(bt_full, cutoff, ticker, best_short, best_long)

    return metrics


# -- Run for both assets --
all_metrics = {}
for ticker in ["GC=F", "CL=F"]:
    metrics = run_asset(ticker)
    all_metrics.update(metrics)


## 8. Results Table
All metrics for both assets, train and test periods, strategy and benchmark.


In [ ]:
df_metrics = pd.DataFrame(all_metrics).T
df_metrics.index.name = "Asset | Period | Type"
df_metrics


## Key Takeaways

- The SMA crossover strategy can reduce drawdowns and volatility by being flat during downtrends.
- Parameter choices found on training data may not generalize well to the test period.
- On gold, the strategy slightly underperformed buy-and-hold on the test set but had lower volatility.
- On crude oil, the strategy showed a negative return on the test set, underperforming buy-and-hold.
- **Past performance does not predict future returns.** This is a simplified educational backtest.
